# 08 · Can a model answer a question about a picture?

Same photo, different question, different answer. That's **visual question answering** (VQA). We'll keep it simple: frozen CLIP vectors for the image and the question, plus a tiny classifier that picks from a short list of answers.

**Read more in [the book](https://arxiv.org/abs/2605.07544):** Chapter 5, Sections 5.2.2, *Visual Question Answering*, and 5.3.2, *VQA Benchmarks and Metrics*.

> **How to use this notebook:** Run the cells top to bottom. When you hit a **Pause and predict**, take a guess before running the next cell. You'll remember way more that way. Cells marked **TODO** are yours to play with: change a value, re-run, and see what happens.

## 1. What exactly are we building?

A deliberately small experiment: 100 VQAv2 rows, the six most common answers, frozen CLIP features, and one little classifier. Small enough that every piece can be checked by eye.

### 1.1 Pick your GPU

Got a GPU (or a few)? Set `GPU_ID` in the next cell to the one you want. Run `nvidia-smi` in a terminal to see the list. Only have one? Leave it at `"0"`. No GPU at all? Set it to `""` and everything runs on the CPU. It's slower, but it works.

This cell has to run *before* PyTorch is imported, so if you change it later, restart the kernel and run from the top.

In [ ]:
# TODO: choose your GPU. This has to run before torch is imported.
#   "0"  -> the first GPU (leave this if you only have one)
#   "1"  -> the second GPU, and so on (`nvidia-smi` lists them)
#   ""   -> no GPU, run everything on the CPU
#   None -> don't restrict anything; PyTorch sees every GPU
GPU_ID = "0"

import os
if GPU_ID is not None:
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # number GPUs the same way nvidia-smi does
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

### 1.2 Grab our tools

Three jobs, three libraries:

- `datasets` gives us the VQAv2 rows, images already decoded.
- `transformers` gives us CLIP and its matching processor.
- `torch` gives us the small classifier and the optimizer to train it.

In [ ]:
from collections import Counter  # for counting answers and votes
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from torch import nn
from torch.nn import functional as F
from transformers import AutoProcessor, AutoModelForZeroShotImageClassification

### 1.3 Pin down the data and the model

We use the same CLIP checkpoint as in the earlier notebooks, plus a 100-row sample of the VQAv2 validation set. A **revision** is a specific saved version on the Hugging Face Hub (think of it like a git commit). Pinning one means the rows and weights can't quietly change under us next month.

In [ ]:
DATASET_ID = "Multimodal-Fatima/VQAv2_sample_validation_100"  # 100 rows from VQAv2 validation
DATASET_REVISION = "572edd7c6cf02a91b9cb17ae7aca363c6205383a"
MODEL_ID = "openai/clip-vit-base-patch32"  # same CLIP as before; pinned so processor and weights never drift
MODEL_REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"

Now the knobs for the experiment:

- `ANSWER_COUNT`: how many answers the classifier can choose from. With 6, a broad VQA problem becomes a six-way multiple choice.
- `TRAIN_STEPS`: how many updates we give the classifier.
- `FUSION_WIDTH`: the size of the classifier's one hidden layer, where image and question information get mixed.

Only the classifier gets trained. CLIP stays frozen the whole time.

In [ ]:
SEED, ANSWER_COUNT, TRAIN_STEPS = 8, 6, 250  # TODO: try ANSWER_COUNT = 10 or TRAIN_STEPS = 50, then Run All
FUSION_WIDTH = 64  # TODO: hidden size of the fusion head; try 16 or 256

The seed fixes two random things: the classifier's starting weights and the shuffle we use to split train from test. It won't make every GPU operation bit-for-bit identical, but it keeps the split the same every time you run the notebook.

> **Heads up:** No GPU? The cell below falls back to the CPU. Everything we train here is small, so that's fine, just a bit slower.

In [ ]:
torch.manual_seed(SEED)  # set before we build the head, so its starting weights repeat
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU if we have one, CPU otherwise

Quick check of where all our tensors are going to live:

In [ ]:
print("device:", device)  # images, questions, and targets all end up here

## 2. What does a VQA record look like?

VQAv2 comes from the paper [*Making the V in VQA Matter*](https://arxiv.org/abs/1612.00837) (Goyal et al., 2017). The authors noticed that models trained on the first VQA dataset could often get the right answer without really looking at the picture. Ask "Is there a ...?" and "yes" is a great guess. So for many of the questions, they collected a second, similar image where the answer is different:

<img src="../assets/diagrams/notebook8/vqav2_fig1.png" alt="Examples from the balanced VQA dataset: the same question with two images and two different answers" width="600">

*Figure 1 from Goyal et al. (2017).*

Same question, two images, two answers. You can't get both right unless you look. Our 100-row sample doesn't keep these pairs together, but the worry is the same, and TODO 2 at the end checks whether our model actually uses the image.

### 2.1 Load the sample

Each row has four things we care about:

- an **image**
- a **question** in plain English
- a **canonical answer**: the single answer the dataset treats as "the" answer
- a list of **answers** from human annotators (VQAv2 collects 10 per question)

The first run downloads the sample. After that it reads from your local cache.

In [ ]:
vqa_data = load_dataset(  # downloads once, then loads from the cache
    DATASET_ID,
    split="validation",
    revision=DATASET_REVISION,
)
print(vqa_data)

### 2.2 Meet our running example

Before any tensors show up, let's just read one row like a person would. We'll start with the question on its own.

In [ ]:
example = vqa_data[0]  # our running example: the very first row
print("question:", example["question"])

> **Pause and predict:** Read that question. What *kind* of answer does it want: a yes/no, a number, a color, a thing? And do you think everyone who answered it said exactly the same words?

The canonical answer is what we'll train on. But the votes tell us something the single label hides: people sometimes phrase the same answer differently, or flat-out disagree when the image is ambiguous.

In [ ]:
vote_counts = Counter(example["answers"])  # how many annotators gave each answer
print("canonical answer / votes:", example["multiple_choice_answer"], dict(vote_counts))

If all the votes agree, great, that's an easy one. If they're split, keep in mind that we're about to train on just one of those answers and treat the rest as wrong.

Next, a quick look at the image itself. We're only checking that we got real, decoded pixels. CLIP's processor will take care of resizing and normalizing later, so we leave the image alone for now.

In [ ]:
print("image:", example["image"].mode, example["image"].size)  # mode and [W, H]: real pixels, not a file path

Now let's see the photo with its question on top. This looks almost too simple to bother with, but it catches a nasty kind of bug: images and questions getting out of step somewhere in your data code. No model can fix that, and it won't crash either. It just trains on nonsense.

Does the answer from the votes make sense for this picture?

In [ ]:
fig, axis = plt.subplots(figsize=(5.5, 3.5))  # the photo, titled with its own question
axis.imshow(example["image"])
axis.set_title(example["question"], wrap=True)
axis.axis("off")
fig.tight_layout()
plt.show()

### 2.3 Which answers is the model allowed to give?

Open-ended VQA can have thousands of different answer strings. That's way too many for 100 rows. So we shrink the problem: keep only the `ANSWER_COUNT` most common canonical answers, so each class shows up a few times instead of once.

Why can such a short list work at all? Because VQA answers are very lopsided. This chart from the original VQA paper ([Antol et al., 2015](https://arxiv.org/abs/1505.00468)) shows the answers people gave for each kind of question. Each bar is one question type, like "what color" or "is there", and each colored band is one answer:

<img src="../assets/diagrams/notebook8/vqa1_fig5.png" alt="Distribution of answers per question type, with and without the image" width="800">

*Figure 5 from Antol et al. (2015).*

Look at "is the", "is there" and "does": almost all yes or no. "How many" is mostly 2, 1, 3 or 4. The bottom half is the interesting part. People were asked to guess the answer *without* seeing the image, and the bars look surprisingly similar. That's the language-prior problem VQAv2 was built to fight, and it's what TODO 2 at the end tests on our own model.

This short list is our **answer vocabulary**: the full menu of things the model can ever say. Notice we count it from the data instead of writing it by hand.

In [ ]:
answer_counts = Counter(vqa_data["multiple_choice_answer"])  # how often does each answer show up?
answer_vocab = tuple(answer for answer, _ in answer_counts.most_common(ANSWER_COUNT))  # keep the most common ones

The classifier outputs one score per answer, in a fixed order. So we need a lookup from each answer string to its column number. Every answer in the vocabulary should get exactly one ID.

In [ ]:
answer_to_id = {answer: index for index, answer in enumerate(answer_vocab)}  # answer string -> classifier column
print("answer vocabulary:", answer_vocab)

Take a good look at that list. Whatever's on it is *everything* this model will ever be able to say.

**TODO:** How much of the dataset did we just throw away? Change `my_top_n` to see more of the answer ranking, and check how many rows our vocabulary actually covers.

In [ ]:
# TODO: change my_top_n to see more (or fewer) of the answer ranking
my_top_n = 15
print("distinct answers in the sample:", len(answer_counts))
print(f"top {my_top_n}:", answer_counts.most_common(my_top_n))
covered = sum(answer_counts[answer] for answer in answer_vocab)
print("rows our vocabulary covers:", covered, "of", len(vqa_data))

Look at what kind of answers sit at the top. Is it mostly yes/no and small numbers? Every row whose answer didn't make the cut gets dropped in the next step. That's a big simplification, and it's one of the main things Notebook 09 gets rid of.

<details>
<summary>Hint</summary>

Want to see which *questions* go with an answer? Try:

```python
[row["question"] for row in vqa_data if row["multiple_choice_answer"] == answer_vocab[0]][:5]
```

</details>

### 2.4 How do we keep the test set honest?

Here's a sneaky one. VQAv2 often asks several questions about the **same photo**. If that photo ended up in both the training set and the test set, a "held-out" score would be partly measuring whether the model remembers a picture it's already seen. Hard to interpret.

The fix: keep only the first eligible question for each image. First, find the rows whose answer is in our vocabulary:

In [ ]:
eligible = [i for i, answer in enumerate(vqa_data["multiple_choice_answer"])  # rows whose answer made the cut
            if answer in answer_to_id]
first_by_image = {}  # image id -> first eligible row that uses it

Then walk through those rows and remember each image ID only the first time we see it. Because we filtered for eligible answers first, every row we keep belongs to our six-answer task.

In [ ]:
for index in eligible:
    first_by_image.setdefault(vqa_data[index]["id_image"], index)  # setdefault only stores the first row per image

`examples` is now our one source of truth for images, questions, and answers. From here on, **row order matters**. Row `i` of every tensor we build has to mean the same image, question, and answer.

In [ ]:
examples = [vqa_data[index] for index in first_by_image.values()]  # dicts keep insertion order, so dataset order survives
print("retained records:", len(examples))

### 2.5 Split into training and held-out rows

We turn each answer into its ID (that's our **target**, the thing the model should predict), then shuffle the row numbers with a seeded random permutation. The first 80% go to training. The rest are **held out**: the classifier never trains on them, so they tell us how it does on questions it hasn't seen.

In [ ]:
targets = torch.tensor([answer_to_id[row["multiple_choice_answer"]] for row in examples])  # answer IDs, in examples order
order = torch.randperm(len(examples), generator=torch.Generator().manual_seed(SEED))  # a seeded shuffle of row numbers

Notice that we don't copy any records. We just keep two lists of row numbers. That way image features, question features, and targets can all share the same row order, and we pick out train or test rows by index.

In [ ]:
train_count = round(0.8 * len(examples))  # 80% train, 20% held out
train_idx, test_idx = order[:train_count], order[train_count:]  # held-out rows never enter the training loop

> **Pause and predict:** We started with 100 rows, then dropped rows outside the vocabulary and repeated photos. Roughly how many held-out rows do you expect? And if there are, say, 10 of them, how much does a single wrong answer move the accuracy?

Always read an accuracy next to its denominator. Keep these two counts in mind for the end of the notebook.

In [ ]:
print("training / held-out rows:", len(train_idx), len(test_idx))  # remember these denominators

Quick math: one mistake costs 100 ÷ (held-out count) percentage points. With 10 held-out rows, that's 10 points per mistake. On a test set this small, accuracy jumps around a lot, so don't read too much into small differences.

## 3. How do images and questions become vectors?

### 3.1 Line up the raw inputs

Images stay as RGB PIL images until CLIP's processor sees them. Questions stay as plain strings until the tokenizer sees them. We build both lists straight from `examples`, so their order matches `targets`.

In [ ]:
images = [row["image"].convert("RGB") for row in examples]  # RGB just in case some images are grayscale; same order as targets

The next cell prints three lengths: images, questions, targets. They should all be the same. If they're not, stop right there. A model can't fix mismatched supervision.

In [ ]:
questions = [row["question"] for row in examples]  # same order as images and targets
print("aligned raw records:", len(images), len(questions), len(targets))

### 3.2 Load CLIP's processor

You've met this one before. The processor knows the exact prep this checkpoint expects: resize, crop, and normalize for images, and tokenization for text. We pin it to the same revision as the model, so the two always match.

In [ ]:
processor = AutoProcessor.from_pretrained(  # image prep + tokenizer for this exact checkpoint
    MODEL_ID, revision=MODEL_REVISION
)

### 3.3 Load CLIP and freeze it

CLIP has an image tower and a text tower, and their outputs live in one shared space. We'll use both towers as fixed **feature extractors**: they turn inputs into vectors, and we never change their weights. With only a handful of training rows, fine-tuning all of CLIP would be asking for trouble.

(`AutoModelForZeroShotImageClassification` sounds like a special model, but for a CLIP checkpoint it just loads the regular CLIP model with both towers.)

In [ ]:
clip = AutoModelForZeroShotImageClassification.from_pretrained(  # loads the full CLIP model: both towers
    MODEL_ID, revision=MODEL_REVISION
).to(device)
clip.eval()  # we're not training CLIP: turn off training-only behavior

Turning off `requires_grad` makes the training boundary official. We can still read features out of CLIP, but no gradient will ever reach its weights. The printed count should be zero.

In [ ]:
for parameter in clip.parameters():  # freeze CLIP: no gradients, no updates
    parameter.requires_grad_(False)
print("trainable CLIP parameters:", sum(p.numel() for p in clip.parameters() if p.requires_grad))

### 3.4 Run the processor on everything

Two calls, one per modality:

- Images get resized and normalized into pixel tensors.
- Questions get tokenized. The processor adds CLIP's start and end tokens, **pads** shorter questions so they all have the same length, and returns an **attention mask** that marks which positions are real tokens and which are padding.

In [ ]:
image_inputs = processor(images=images, return_tensors="pt")  # resize, crop, normalize; row order kept
question_inputs = processor(  # tokenize, pad to the longest question, return an attention mask
    text=questions, padding=True, truncation=True, return_tensors="pt"
)

Everything moves to our device together. Moving doesn't reorder anything, so row `i` still lines up across all three.

In [ ]:
image_inputs = {name: value.to(device) for name, value in image_inputs.items()}  # same rows, now on our device
question_inputs = {name: value.to(device) for name, value in question_inputs.items()}
targets = targets.to(device)

What to look for in the shapes:

- The pixel tensor should be `[number of examples, 3, 224, 224]`.
- The question IDs should be `[number of examples, longest question length in tokens]`.
- The first number should be the same everywhere. The rest are different because images and text are different things, and that's fine.

In [ ]:
print("pixel batch:", tuple(image_inputs["pixel_values"].shape))  # [N, 3, 224, 224]
print("question IDs:", tuple(question_inputs["input_ids"].shape))  # [N, tokens]
print("answer rows:", len(targets))

## 4. How do we combine what the image shows with what the question asks?

### 4.1 Encode each image once

CLIP turns each photo into one vector. We normalize it to length 1 (just like in Notebook 01), so the length of the vector can't sneak in as a clue. The classifier has to work with *direction*: which features are high and which are low.

Because CLIP is frozen, these vectors never change. So we compute them **once**, right here, and reuse them for every training step.

In [ ]:
with torch.no_grad():  # CLIP is frozen, so we compute these once and reuse them
    image_output = clip.get_image_features(**image_inputs)
    image_features = F.normalize(image_output.pooler_output, dim=-1)  # one length-1 vector per image

### 4.2 Encode each question once

Same deal for text. The text tower turns each whole question into one vector in the same shared CLIP space. Computed once, reused for all `TRAIN_STEPS` updates.

In [ ]:
with torch.no_grad():  # frozen too: one vector per question, computed once
    question_output = clip.get_text_features(**question_inputs)
    question_features = F.normalize(question_output.pooler_output, dim=-1)

### 4.3 Build the fusion head

**Fusion** just means combining information from two modalities (here, image and text). We'll do the simplest version there is:

1. **Concatenate** the image vector and the question vector side by side, so `[N, D]` and `[N, D]` become `[N, 2D]`.
2. Pass that through a small two-layer classifier: a linear layer down to `FUSION_WIDTH`, a ReLU, and a linear layer out to one score per answer.

Real VQA models do something far fancier. This one is deliberately simple, so you can see the whole thing at once.

> **Something to chew on:** Say two rows share an image but ask different questions. Which half of the concatenated vector changes? Now flip it: same question, different image. Concatenation keeps both clues intact, but it leaves *all* the work of relating them to that one small hidden layer.

In [ ]:
class MiniVQA(nn.Module):  # the only part we train
    def __init__(self, feature_width, answer_count):
        super().__init__()
        self.answer_head = nn.Sequential(
            nn.Linear(feature_width * 2, FUSION_WIDTH),  # 2D -> FUSION_WIDTH
            nn.ReLU(),
            nn.Linear(FUSION_WIDTH, answer_count),  # FUSION_WIDTH -> one score per answer
        )
    def forward(self, image_state, question_state):
        fused = torch.cat((image_state, question_state), dim=-1)  # [N, D] + [N, D] -> [N, 2D]
        return self.answer_head(fused)

### 4.4 Smoke-test the head

Before training, let's make sure the plumbing works. We feed in two image–question pairs and check what comes out. The raw scores a classifier outputs are called **logits**: one number per answer, and the biggest one wins.

In [ ]:
feature_width = image_features.shape[-1]  # read the width from CLIP instead of hard-coding it
model = MiniVQA(feature_width, len(answer_vocab)).to(device)
with torch.inference_mode():  # quick test on the first two rows
    sample_logits = model(image_features[:2], question_features[:2])

> **Pause and predict:** Two rows in, `ANSWER_COUNT` answers to score. What shape will `sample_logits` have? And how will the head's parameter count compare with CLIP's roughly 150 million?

In [ ]:
print("logits [record, answer]:", tuple(sample_logits.shape))  # one row per pair, one column per answer
print("trainable head parameters:", sum(parameter.numel() for parameter in model.parameters()))  # just the head; CLIP isn't inside model

`[2, 6]` with the default settings: two rows, six scores each, one per answer in `answer_vocab`. And the parameter count only covers the small head. It's tiny next to CLIP, which is the whole point: we're reusing CLIP's representations and only learning the task-specific bit.

## 5. Does the head actually learn to answer?

### 5.1 Train on the training rows only

We use **cross-entropy** loss, the standard loss for picking one class out of many. It pushes up the score of the correct answer and pushes down the others.

One thing to notice in the loop: it only ever indexes `train_idx`. The held-out features are sitting right there in memory, but they never touch the loss.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-3)  # only the head's parameters
loss_history = []

Each step feeds *all* the training rows through the head at once (that's called full-batch training, and it's fine when the data is this small). Since the CLIP features are precomputed, the whole loop takes no time at all. We record the loss at every step so we can plot it.

In [ ]:
model.train()  # full batch: every step sees all training rows
for _ in range(TRAIN_STEPS):
    logits = model(image_features[train_idx], question_features[train_idx])
    loss = F.cross_entropy(logits, targets[train_idx])  # training rows only
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())

The loss curve answers one narrow question: can the head fit the training rows? A curve that goes down is good news, but it says **nothing** yet about questions the model hasn't seen.

In [ ]:
fig, axis = plt.subplots(figsize=(6, 3.2))  # training loss only: no info about held-out rows yet
axis.plot(loss_history)
axis.set(xlabel="update", ylabel="cross-entropy", title="Mini VQA training loss")
axis.grid(alpha=0.25)
fig.tight_layout()
plt.show()

### 5.2 One prediction rule for both splits

This little helper takes a list of row numbers and returns the highest-scoring answer ID for each. It runs in inference mode (no gradients tracked) and switches the head to eval mode first. Our head has no dropout, so eval mode doesn't change anything here, but it's a good habit.

In [ ]:
@torch.inference_mode()  # no gradients needed for predictions
def predict(indices):
    model.eval()
    return model(image_features[indices], question_features[indices]).argmax(dim=-1)  # highest score wins

Now we predict on both splits, using the exact same rule.

> **Pause and predict:** Which split will score higher, training or held-out? By a little, or by a lot? Remember: the head has seen every training pair `TRAIN_STEPS` times, and the held-out pairs never.

In [ ]:
train_predictions = predict(train_idx)  # same rule for both splits
test_predictions = predict(test_idx)

Accuracy is simple: compare each prediction with its target (that gives `True` or `False`), turn those into 1s and 0s, and take the average.

In [ ]:
train_accuracy = train_predictions.eq(targets[train_idx]).float().mean()  # fraction of rows we got right
test_accuracy = test_predictions.eq(targets[test_idx]).float().mean()

Read these two numbers together. A high training accuracy with a much lower held-out accuracy means the head mostly **memorized** its training rows rather than learning something that carries over. That's not a working VQA system. And remember the denominator: with this few held-out rows, the held-out number is noisy.

In [ ]:
print("training accuracy:", f"{train_accuracy:.1%}")  # how well we fit what we've seen
print("held-out accuracy:", f"{test_accuracy:.1%}")  # how well that carries over

### 5.3 Look at the actual mistakes

Accuracy tells you *how often* the model was right. Examples tell you *how* it was wrong. For each printed row, read the question, the target, and the prediction.

Some things to look for: does it keep guessing the most common answer? Are the misses on genuinely ambiguous images? Or did it miss a distinction in the question that our simple concatenation head just never picked up?

In [ ]:
for position, index in enumerate(test_idx[:5].tolist()):  # TODO: show more than 5 held-out rows
    target = answer_vocab[targets[index].item()]
    prediction = answer_vocab[test_predictions[position].item()]
    print("question:", examples[index]["question"])
    print("target / prediction:", target, "/", prediction)

## Recap

That's the whole trip. Our VQA model is three pieces stuck together, and only the last one learns:

```text
image     -> CLIP image tower (frozen) -> normalize -> [N, D] ─┐
                                                               ├─ concat -> [N, 2D]
question  -> CLIP text tower  (frozen) -> normalize -> [N, D] ─┘
      ↓ Linear -> ReLU -> Linear   (MiniVQA, the only trained part)
[N, ANSWER_COUNT] scores -> argmax -> one answer from answer_vocab
```

**The big ideas:**

- VQA takes an image *and* a question. The question decides which part of the image matters.
- Picking from a **fixed answer vocabulary** turns VQA into plain classification. Simple, but the model can never say anything outside that list.
- Frozen encoders mean we compute features **once** and train only a small head on top. That keeps "reusing a representation" separate from "learning the task".
- Keep splits honest: one question per photo, so a photo can't sit in both train and test.
- On a tiny test set, always read accuracy next to its denominator, and look at individual mistakes.

### Check your understanding

1. Why do we keep only one question per image before splitting?
2. A held-out question's correct answer is "red", but "red" isn't in `answer_vocab`. What can MiniVQA possibly say, and could it ever get this one right?
3. Why can we compute the CLIP features once, outside the training loop, instead of on every step?

<details>
<summary>Show answers</summary>

1. If the same photo showed up in training and held-out rows, the held-out score would partly reflect remembering that photo, not answering new questions.
2. It can only ever output one of the `ANSWER_COUNT` answers in `answer_vocab`, so it can never get "red" right. In this notebook that row would have been filtered out before the split, which is also why our accuracy only measures the six-answer task.
3. CLIP is frozen (`requires_grad` is off and the optimizer only holds the head's parameters), so the same input always gives the same features. Recomputing them every step would just waste time.

</details>

## Try it yourself

### TODO 1: Ask your own question about a held-out image

`predict` only works with questions that are already in the dataset. To ask *your* question, we encode it with CLIP's text tower the exact same way as before, then feed it into the trained head next to a held-out image's features. The cell also shows what `predict` says for the dataset's own question, so you can compare.

Change `my_position` to pick a different held-out image, and change `my_question` to ask something else. Before you run it, guess: which answer will win?

In [ ]:
# TODO: pick a held-out image (0 to len(test_idx) - 1) and write your own question
my_position = 0
my_question = "Is there a person in this picture?"

my_index = test_idx[my_position].item()  # row number in examples
my_question_inputs = processor(text=[my_question], padding=True, truncation=True, return_tensors="pt")
my_question_inputs = {name: value.to(device) for name, value in my_question_inputs.items()}
with torch.no_grad():  # same encoding as the dataset questions
    my_question_output = clip.get_text_features(**my_question_inputs)
    my_question_features = F.normalize(my_question_output.pooler_output, dim=-1)

dataset_answer = answer_vocab[predict(test_idx[my_position:my_position + 1]).item()]
model.eval()
with torch.inference_mode():
    my_probabilities = model(image_features[my_index:my_index + 1], my_question_features).softmax(dim=-1)[0]

fig, axis = plt.subplots(figsize=(4, 3))
axis.imshow(examples[my_index]["image"])
axis.set_title(my_question, wrap=True)
axis.axis("off")
fig.tight_layout()
plt.show()

print("dataset question:", examples[my_index]["question"], "-> predicted:", dataset_answer)
print("your question:", my_question)
for answer, probability in sorted(zip(answer_vocab, my_probabilities.tolist()), key=lambda pair: -pair[1]):
    print(f"  {answer:>12}: {probability:.1%}")

`softmax` turns the six logits into probabilities that add up to 1, so you can see how sure the head is. Try a question whose real answer isn't in `answer_vocab` at all. The model will still pick something from its list, with no way of saying "none of these".

<details>
<summary>Hint</summary>

Want to check how many held-out images there are? `len(test_idx)`. And `print(answer_vocab)` reminds you which answers are even possible. Questions phrased like the dataset's own are probably your best bet, since those are the only kind the head has ever trained on.

</details>

### TODO 2: Is the model even looking at the image?

VQA datasets are famous for this: a model can often guess well from the question alone. ("Is there a...?" is usually "yes".) Let's test ours. The cell below keeps every held-out question, but pairs it with the **wrong** image by shifting the image features one row over.

In [ ]:
# TODO: try torch.zeros_like(image_features[test_idx]) instead, to give the head no image at all
wrong_images = image_features[test_idx].roll(1, dims=0)  # each question now gets a different held-out image
model.eval()
with torch.inference_mode():
    wrong_predictions = model(wrong_images, question_features[test_idx]).argmax(dim=-1)
wrong_accuracy = wrong_predictions.eq(targets[test_idx]).float().mean()
print("held-out accuracy, right images:", f"{test_accuracy:.1%}")
print("held-out accuracy, wrong images:", f"{wrong_accuracy:.1%}")

If the two numbers are close, the head is leaning mostly on the question. If accuracy drops a lot, the image really matters. Just remember how small the held-out set is before you draw big conclusions.

### TODO 3: Grade it the way VQAv2 does

We scored the model against one canonical answer. VQAv2's own metric is more forgiving: an answer counts as fully right if at least 3 annotators gave it, and partly right if fewer did. The cell below uses a simplified version of that rule, `min(votes / 3, 1)`. (The official metric also averages over subsets of annotators and cleans up answer strings, so expect it to differ a little.)

In [ ]:
vqa_style_scores = []
for position, index in enumerate(test_idx.tolist()):
    prediction = answer_vocab[test_predictions[position].item()]
    votes = Counter(examples[index]["answers"])[prediction]  # how many annotators said our answer
    vqa_style_scores.append(min(votes / 3, 1.0))
print("held-out accuracy, canonical answer:", f"{test_accuracy:.1%}")
print("held-out accuracy, VQA-style:", f"{sum(vqa_style_scores) / len(vqa_style_scores):.1%}")

Did the score go up? Any gap comes from predictions that weren't the canonical answer, but that enough people said anyway. That's the vote counts from Section 2.2 paying off.

<details>
<summary>Hint</summary>

Want to see which rows got partial credit? Print `examples[index]["question"]`, the prediction, and `dict(Counter(examples[index]["answers"]))` inside the loop, for any row where the score is between 0 and 1.

</details>

---

**Up next, Notebook 09:** no more fixed answer list. We'll build a LLaVA-style model that projects CLIP's image patches into a real causal language model, so the answer gets *generated* token by token, just like a chat reply.